# ✈️ Plane Type Classifier — How to Use

This script identifies the model of an aircraft from an image using a trained deep learning model. It supports two model formats: **fastai** (`.pkl`) and **PyTorch** (`.pth`).

---

## Requirements

Install the required libraries before running:

```bash
# For PyTorch models (ResNet-50 trained with torchvision):
pip install torch torchvision pillow

# For fastai models (.pkl export):
pip install fastai
```

---

## Setup

### 1. Set your model path

At the top of the code cell, update `MODEL_PATH` to point to your trained model file:

```python
MODEL_PATH = "resnet50_aircraft_classifier.pth"  # your .pth or .pkl file
```

### 2. For PyTorch models — add a `class_names.txt`

If using a `.pth` model (e.g., the ResNet-50 trained in the Colab notebook), create a `class_names.txt` file in the same directory — one aircraft class per line, in the same order used during training.

Example first few lines:
```
707-320
727-200
737-200
737-300
...
```

The FGVC-Aircraft dataset has **100 classes** in alphabetical order. For fastai `.pkl` models, class names are embedded in the model — no extra file needed.

---

## Running the Script

1. Run the code cell (Shift+Enter).
2. 2. When prompted, type the **full path** to a plane image:
   3.    ```
      Enter path to plane image: /path/to/your/photo.jpg
      ```
      3. The script prints the predicted aircraft type, confidence score, and top-5 predictions.
      4. Type `q` or `quit` to exit the loop.

      ---

      ## Example Output

      ```
      ═══════════════════════════════════════════════════════
    Image      : /home/user/boeing737.jpg
    ✈ Plane type : 737-800
    Confidence  : 91.4%
   ──────────────────────────────────────────────────────
    Top predictions:
     1. 737-800  91.4%  ██████████████████
     2.   2. 737-700   5.2%  █
          3.   3. 737-900   2.1%
               4.   4. 737-600   0.8%
                    5.   5. 757-200   0.5%
                         6. ═══════════════════════════════════════════════════════
                         7. ```

                         ---

                         ## Notes

                         - Images are automatically resized and center-cropped to **224×224** with ImageNet normalization — the same preprocessing used during training.
                         - - The script tries fastai first; if the model file ends in `.pth` or `.pt`, it falls back to the bare PyTorch loader.
                           - - The model architecture assumed for PyTorch fallback is **ResNet-50** with a 100-class output head (matching the FGVC-Aircraft training setup).

In [ ]:
from pathlib import Path
import sys

# ── Plane Classifier ──────────────────────────────────────────────────────────
# Loads a trained fastai or PyTorch model and predicts the plane type
# from a user-supplied image path.
# ─────────────────────────────────────────────────────────────────────────────

MODEL_PATH = "planes_model.pkl"   # ← change to your model file if different

# ──────────────────────────────────────────────────────────────────────────────
# 1. Try to load a fastai learner (.pkl export)
# ──────────────────────────────────────────────────────────────────────────────
def load_fastai_learner(model_path: str):
        """Load a model exported with learn.export()."""
        try:
                    from fastai.vision.all import load_learner
                    learn = load_learner(model_path)
                    return learn
except ImportError:
        return None
except Exception as e:
        print(f"[fastai] Could not load model: {e}")
        return None

# ──────────────────────────────────────────────────────────────────────────────
# 2. Predict with fastai learner
# ──────────────────────────────────────────────────────────────────────────────
def predict_fastai(learn, image_path: str):
        """Run inference with a fastai learner and return (label, confidence)."""
        from fastai.vision.all import PILImage
        img = PILImage.create(image_path)
        pred_class, pred_idx, probs = learn.predict(img)
        confidence = float(probs[pred_idx]) * 100
        return str(pred_class), confidence, {
            cls: float(p) * 100
            for cls, p in zip(learn.dls.vocab, probs)
        }

    # ──────────────────────────────────────────────────────────────────────────────
    # 3. Fallback: bare PyTorch model (.pth / .pt) with a class_names.txt
    # ──────────────────────────────────────────────────────────────────────────────
    def load_pytorch_model(model_path: str, class_names_path: str = "class_names.txt"):
            """
                Fallback for a plain torchvision model saved with torch.save().
                    Expects a 'class_names.txt' file (one class per line) alongside the model.
                        """
            try:
                        import torch
                        import torchvision.models as models

                # Load class names
                        names_file = Path(class_names_path)
                        if not names_file.exists():
                                        print("[PyTorch] class_names.txt not found – cannot use PyTorch fallback.")
                                        return None, None

                        class_names = names_file.read_text().strip().splitlines()
                        num_classes = len(class_names)

                # Try ResNet-50 architecture (common default)
                        model = models.resnet50(weights=None)
                        model.fc = torch.nn.Linear(model.fc.in_features, num_classes)
                        model.load_state_dict(torch.load(model_path, map_location="cpu"))
                        model.eval()
                        return model, class_names
except Exception as e:
        print(f"[PyTorch] Could not load model: {e}")
        return None, None

def predict_pytorch(model, class_names, image_path: str):
        """Run inference with a torchvision model."""
        import torch
        from torchvision import transforms
        from PIL import Image

        transform = transforms.Compose([
            transforms.Resize(256),
            transforms.CenterCrop(224),
            transforms.ToTensor(),
            transforms.Normalize([0.485, 0.456, 0.406],
                                 [0.229, 0.224, 0.225]),
        ])

        img = Image.open(image_path).convert("RGB")
        tensor = transform(img).unsqueeze(0)

        with torch.no_grad():
                    outputs = model(tensor)
                    probs = torch.softmax(outputs, dim=1)[0]

        top_idx = int(probs.argmax())
        confidence = float(probs[top_idx]) * 100
        label = class_names[top_idx]
        all_probs = {cls: float(p) * 100 for cls, p in zip(class_names, probs)}
        return label, confidence, all_probs

    # ──────────────────────────────────────────────────────────────────────────────
    # 4. Pretty-print results
    # ──────────────────────────────────────────────────────────────────────────────
    def display_results(image_path: str, label: str, confidence: float, all_probs: dict):
            print("\n" + "═" * 55)
            print(f"  Image   : {image_path}")
            print(f"  ✈  Plane type : {label}")
            print(f"  Confidence    : {confidence:.1f}%")
            print("─" * 55)
            # Show top-5 predictions sorted by probability
            top5 = sorted(all_probs.items(), key=lambda x: x[1], reverse=True)[:5]
    print("  Top predictions:")
    for rank, (cls, prob) in enumerate(top5, 1):
                bar = "█" * int(prob / 5)
                print(f"    {rank}. {cls:<30} {prob:5.1f}%  {bar}")
            print("═" * 55 + "\n")

# ──────────────────────────────────────────────────────────────────────────────
# 5. Main interactive loop
# ──────────────────────────────────────────────────────────────────────────────
def main():
        model_file = Path(MODEL_PATH)

        # --- Load model (fastai first, then PyTorch fallback) ---
        learn = None
    pt_model, class_names = None, None

    if model_file.suffix == ".pkl":
                learn = load_fastai_learner(str(model_file))
                if learn:
                                print(f"✅  Loaded fastai model from '{model_file}'")
                                print(f"    Classes: {list(learn.dls.vocab)}\n")
    elif model_file.suffix in (".pth", ".pt"):
                pt_model, class_names = load_pytorch_model(str(model_file))
                if pt_model:
                                print(f"✅  Loaded PyTorch model from '{model_file}'")
                                print(f"    Classes: {class_names}\n")

            if learn is None and pt_model is None:
                        print(
                                        f"\n❌  No model could be loaded from '{MODEL_PATH}'.\n"
                                        "    Make sure the file exists and the correct library is installed.\n"
            "    • For fastai models (.pkl) : pip install fastai\n"
            "    • For PyTorch models (.pth): pip install torch torchvision\n"
)
        return

    # --- Interactive prediction loop ---
    print("Plane Type Classifier — type 'q' to quit\n")
    while True:
                image_path = input("Enter path to plane image: ").strip()

                if image_path.lower() in ("q", "quit", "exit"):
                                print("Goodbye! ✈")
                                break

                if not Path(image_path).exists():
                                print(f"  ⚠  File not found: '{image_path}'. Try again.\n")
                                continue

                try:
                                if learn is not None:
                                                    label, confidence, all_probs = predict_fastai(learn, image_path)
                else:
                                    label, confidence, all_probs = predict_pytorch(pt_model, class_names, image_path)

                    display_results(image_path, label, confidence, all_probs)
except Exception as e:
            print(f"  ❌  Prediction failed: {e}\n")

main()